# Time Series Statistics

Notebook for **Chapter 10** of the *stats* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/stats/ch10.html

Tip: change a number, predict the output, then run the cell again.

## 10.1 What Makes a Time Series Different: Trend, Season, Noise

## 10.2 Stationarity and Differencing

## 10.3 ACF and PACF: Reading the Memory of a Series

## 10.4 ARIMA and SARIMA

In [ ]:
# not run here: statsmodels is not installed in this environment
from statsmodels.tsa.statespace.sarimax import SARIMAX
m = SARIMAX(train, order=(0,1,1), seasonal_order=(0,1,1,12)).fit()
print(m.summary())                 # coefficients, AIC, BIC, Ljung-Box
fc = m.get_forecast(12)
fc.predicted_mean, fc.conf_int()   # forecast and interval

## 10.5 Exponential Smoothing

## 10.6 Forecast Evaluation

## 10.7 Is the Model Finished? Residual Diagnostics and the Ljung-Box Test

A forecasting model is finished when its residuals look like white noise: no pattern in the mean, constant spread, no autocorrelation. Any leftover autocorrelation is information the model failed to use. Look at the residual ACF first, then run a formal test.

In [ ]:
import numpy as np
from scipy.stats import chi2
rng = np.random.default_rng(7)
n, phi = 300, 0.6
e = rng.normal(0, 1, n); y = np.zeros(n)
for t in range(1, n): y[t] = phi * y[t-1] + e[t]
def acf(x, k):
    x = x - x.mean(); d = np.dot(x, x)
    return np.array([np.dot(x[:len(x)-l], x[l:]) / d for l in range(1, k + 1)])
def ljung_box(res, h, n_params=0):
    r = acf(res, h); N = len(res)
    Q = N * (N + 2) * np.sum(r ** 2 / (N - np.arange(1, h + 1)))
    return Q, 1 - chi2.cdf(Q, h - n_params)
res_mean = y - y.mean()                                       # model 1: "just a mean"
phi_hat = np.dot(y[1:] - y.mean(), y[:-1] - y.mean()) / np.dot(y[:-1] - y.mean(), y[:-1] - y.mean())
res_ar = (y[1:] - y.mean()) - phi_hat * (y[:-1] - y.mean())   # model 2: AR(1)
print("residual ACF at lags 1-3, mean-only model:", np.round(acf(res_mean, 3), 3))
print("residual ACF at lags 1-3, AR(1) model    :", np.round(acf(res_ar, 3), 3))
print(f"phi_hat = {phi_hat:.3f} (true 0.6); 95% white-noise band for the ACF = +/- {1.96/np.sqrt(n):.3f}")
Q, p = ljung_box(res_mean, 10); print(f"Ljung-Box, mean-only, 10 lags: Q = {Q:.1f}, p < 0.0001" if p < 1e-4 else f"Ljung-Box, mean-only: Q = {Q:.1f}, p = {p:.3f}")
Q, p = ljung_box(res_ar, 10, 1); print(f"Ljung-Box, AR(1),     10 lags: Q = {Q:.1f}, p = {p:.3f}  (df = 10 - 1 fitted parameter)")

residual ACF at lags 1-3, mean-only model: [0.607 0.384 0.27 ]
residual ACF at lags 1-3, AR(1) model    : [-0.016 -0.011  0.054]
phi_hat = 0.607 (true 0.6); 95% white-noise band for the ACF = +/- 0.113
Ljung-Box, mean-only, 10 lags: Q = 208.1, p < 0.0001
Ljung-Box, AR(1),     10 lags: Q = 10.8, p = 0.289  (df = 10 - 1 fitted parameter)


## 10.8 Prediction Intervals and How They Grow

A point forecast with no interval is half an answer. The interval should widen as you look further ahead, but how fast depends on the model. For an AR(1) with coefficient φ and shock SD σ, the h-step forecast error has variance σ² (1 + φ² + … + φ2(h−1)), which levels off at the long-run variance σ² / (1 − φ²). For a random walk, the variance is σ² h, which grows forever. Intervals from differenced (integrated) models keep growing; intervals from stationary models stop.

In [ ]:
import numpy as np
rng = np.random.default_rng(103)
phi, sigma, mu = 0.8, 2.0, 50.0
hs = [1, 2, 5, 10, 20]
print("Forecast error SD at horizon h")
print(" h   AR(1) formula   simulated   random-walk formula")
reps = 200000
x0 = mu                                                 # last observation sits at the mean
e = rng.normal(0, sigma, (reps, max(hs)))
paths = np.zeros((reps, max(hs))); prev = np.full(reps, x0)
for j in range(max(hs)):
    prev = mu + phi * (prev - mu) + e[:, j]; paths[:, j] = prev
for h in hs:
    f = sigma * np.sqrt(sum(phi ** (2 * j) for j in range(h)))
    print(f"{h:>2}   {f:>12.3f}   {paths[:, h-1].std():>9.3f}   {sigma*np.sqrt(h):>14.3f}")
print("long-run SD of a stationary AR(1):", round(sigma / np.sqrt(1 - phi ** 2), 3))
print("95% interval for h=10 from AR(1): 50 +/-", round(1.96 * sigma * np.sqrt(sum(phi ** (2 * j) for j in range(10))), 2),
      "  vs random walk: +/-", round(1.96 * sigma * np.sqrt(10), 2))

Forecast error SD at horizon h
 h   AR(1) formula   simulated   random-walk formula
 1          2.000       1.990            2.000
 2          2.561       2.551            2.828
 5          3.149       3.152            4.472
10          3.314       3.312            6.325
20          3.333       3.341            8.944
long-run SD of a stationary AR(1): 3.333
95% interval for h=10 from AR(1): 50 +/- 6.5   vs random walk: +/- 12.4


## 10.9 Rolling-Origin Cross-Validation: A Full Worked Comparison

Section 10.6 introduced forecast error measures. Here is how to use them to choose among models without fooling yourself. Ordinary cross-validation shuffles rows, which lets the model peek at the future. Time series instead use a **rolling origin**: pick a cutoff, train on everything before it, forecast the next H steps, score, then move the cutoff forward and repeat. With an *expanding window* you keep all history; with a *sliding window* you keep only the last W points, which is better when the process changes.

In [ ]:
import numpy as np
rng = np.random.default_rng(102)
T, m = 144, 12
t = np.arange(T)
season = 10 * np.sin(2 * np.pi * t / m) + 4 * np.cos(2 * np.pi * t / m * 2)
y = 100 + 0.5 * t + season + rng.normal(0, 3, T)             # trend + seasonality + noise
H = 12
def f_naive(tr, h):  return np.repeat(tr[-1], h)
def f_snaive(tr, h): return np.array([tr[len(tr) - m + (i % m)] for i in range(h)])
def f_drift(tr, h):  return tr[-1] + (tr[-1] - tr[0]) / (len(tr) - 1) * np.arange(1, h + 1)
def f_reg(tr, h):                                              # linear trend + 11 month dummies, fit by least squares
    def X(idx): return np.column_stack([np.ones(len(idx)), idx] + [(idx % m == j).astype(float) for j in range(1, m)])
    idx = np.arange(len(tr)); b = np.linalg.lstsq(X(idx), tr, rcond=None)[0]
    return X(np.arange(len(tr), len(tr) + h)) @ b
models = {"naive": f_naive, "seasonal naive": f_snaive, "drift": f_drift, "trend + seasonal dummies": f_reg}
origins = range(72, T - H + 1, 6)                              # rolling origin, expanding window, 12 origins
print("origins:", len(list(origins)), " horizon:", H)
mae = {k: [] for k in models}
for o in origins:
    tr, te = y[:o], y[o:o + H]
    for k, f in models.items(): mae[k].append(np.mean(np.abs(te - f(tr, H))))
scale = np.mean(np.abs(y[m:72] - y[:72 - m]))                  # MASE scale: in-sample seasonal naive MAE on the first 72 points
print(f"{'model':<26}{'mean MAE':>9}{'worst origin':>14}{'MASE':>7}")
for k in models: print(f"{k:<26}{np.mean(mae[k]):>9.2f}{np.max(mae[k]):>14.2f}{np.mean(mae[k])/scale:>7.2f}")

origins: 11  horizon: 12
model                      mean MAE  worst origin   MASE
naive                          8.57         10.91   1.35
seasonal naive                 6.15          8.09   0.97
drift                          8.67         13.46   1.37
trend + seasonal dummies       2.70          3.93   0.43


## 10.10 Regression with Autocorrelated Errors (ARIMAX)

When a series depends on outside drivers (promotions, temperature, price), you want to estimate their effect and still respect that the leftover noise has memory. The model is yt = β0 + β1 xt + ut where ut follows an ARMA process. If you ignore the memory and use ordinary least squares, the coefficient estimate is still unbiased, but its standard error is wrong, usually too small when both the driver and the errors are positively autocorrelated. You then see "significant" effects that are not.

In [ ]:
import numpy as np
rng = np.random.default_rng(104)
n, rho, reps = 120, 0.7, 1000
def one():
    promo = np.zeros(n)
    for s in range(0, n, 10): promo[s:s+5] = 1                   # promo runs in 5-day blocks (also autocorrelated)
    e = rng.normal(0, 1, n); u = np.zeros(n)
    for t in range(1, n): u[t] = rho * u[t-1] + e[t]
    y = 50 + 3.0 * promo + u
    X = np.column_stack([np.ones(n), promo])
    b = np.linalg.solve(X.T @ X, X.T @ y); r = y - X @ b
    se_ols = np.sqrt(np.diag(np.linalg.inv(X.T @ X)) * (r @ r) / (n - 2))[1]
    # Cochrane-Orcutt: estimate rho from OLS residuals, transform, refit once more
    rh = np.dot(r[1:], r[:-1]) / np.dot(r[:-1], r[:-1])
    ys, Xs = y[1:] - rh * y[:-1], X[1:] - rh * X[:-1]
    bs = np.linalg.solve(Xs.T @ Xs, Xs.T @ ys); rs = ys - Xs @ bs
    se_co = np.sqrt(np.diag(np.linalg.inv(Xs.T @ Xs)) * (rs @ rs) / (n - 3))[1]
    return b[1], se_ols, bs[1], se_co
out = np.array([one() for _ in range(reps)])
print("promo effect (truth 3.0), 1000 simulated datasets")
print(f"OLS            : mean estimate {out[:,0].mean():.3f}, actual SD of estimates {out[:,0].std():.3f}, average reported SE {out[:,1].mean():.3f}, 95% CI coverage {np.mean(np.abs(out[:,0]-3) < 1.96*out[:,1]):.3f}")
print(f"AR(1) errors CO: mean estimate {out[:,2].mean():.3f}, actual SD of estimates {out[:,2].std():.3f}, average reported SE {out[:,3].mean():.3f}, 95% CI coverage {np.mean(np.abs(out[:,2]-3) < 1.96*out[:,3]):.3f}")

promo effect (truth 3.0), 1000 simulated datasets
OLS            : mean estimate 3.002, actual SD of estimates 0.279, average reported SE 0.246, 95% CI coverage 0.916
AR(1) errors CO: mean estimate 3.003, actual SD of estimates 0.225, average reported SE 0.229, 95% CI coverage 0.959


## 10.11 ETS in One Table, and Intermittent Demand

Exponential smoothing (section 10.5) is a family. The **ETS** framework names a model by three letters: Error (additive A or multiplicative M), Trend (none N, additive A, damped Ad) and Seasonal (N, A or M). ETS(A,N,N) is simple exponential smoothing, ETS(A,A,N) is Holt's linear method, ETS(A,Ad,A) is damped-trend Holt-Winters with additive seasonality. Software fits all candidates, picks by AICc, and gives proper prediction intervals.

In [ ]:
import numpy as np
rng = np.random.default_rng(106)
T = 400; p_occ = 0.25
demand = np.where(rng.random(T) < p_occ, rng.poisson(4, T) + 1, 0)       # mean per period = 0.25 * 5 = 1.25
print("share of zero periods:", round((demand == 0).mean(), 2), " true mean per period: 1.25  sample mean:", round(demand.mean(), 2))
def ses(d, a=0.1):
    f = np.empty(len(d)); l = d[:10].mean()
    for t, x in enumerate(d): f[t] = l; l = l + a * (x - l)
    return f                                                               # f[t] = forecast made before seeing d[t]
def croston(d, a=0.1, sba=False):
    f = np.empty(len(d)); nz = np.flatnonzero(d)
    z = d[nz[0]]; q = nz[0] + 1.0; last = nz[0]
    for t, x in enumerate(d):
        f[t] = (1 - a / 2 if sba else 1) * z / q if t > nz[0] else d[:10].mean()
        if x > 0:
            z = z + a * (x - z); q = q + a * ((t - last) - q); last = t
    return f
s, c, b = ses(demand), croston(demand), croston(demand, sba=True)
sl = slice(100, None)
print(f"{'method':<14}{'mean forecast':>14}{'MAE':>7}{'RMSE':>7}")
for name, f in [("SES a=0.1", s), ("Croston", c), ("Croston SBA", b), ("always zero", np.zeros(T))]:
    e = demand[sl] - f[sl]; print(f"{name:<14}{f[sl].mean():>14.2f}{np.abs(e).mean():>7.2f}{np.sqrt((e**2).mean()):>7.2f}")
print("SD of the forecast over time: SES", round(s[sl].std(), 2), " Croston", round(c[sl].std(), 2))

share of zero periods: 0.75  true mean per period: 1.25  sample mean: 1.24
method         mean forecast    MAE   RMSE
SES a=0.1               1.19   1.74   2.26
Croston                 1.23   1.79   2.25
Croston SBA             1.17   1.76   2.25
always zero             0.00   1.16   2.51
SD of the forecast over time: SES 0.62  Croston 0.3


## 10.12 Changing Variance: A First Look at GARCH

Financial returns, sensor noise and demand shocks often show **volatility clustering**: calm weeks followed by wild ones. The returns themselves look uncorrelated, but their squares are not. The GARCH(1,1) model describes this with two equations: rt = σt εt and σt² = ω + α rt−1² + β σt−1². Today's variance is a baseline plus a reaction to yesterday's surprise plus memory of yesterday's variance. Persistence is α + β; the long-run variance is ω / (1 − α − β).

In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import chi2
rng = np.random.default_rng(105)
n = 2500; om, al, be = 0.05, 0.10, 0.85          # long-run variance = om / (1 - al - be) = 1.0
r = np.zeros(n); s2 = np.zeros(n); s2[0] = om / (1 - al - be)
for t in range(n):
    if t > 0: s2[t] = om + al * r[t-1] ** 2 + be * s2[t-1]
    r[t] = np.sqrt(s2[t]) * rng.normal()
def acf(x, k):
    x = x - x.mean(); d = np.dot(x, x); return np.array([np.dot(x[:len(x)-l], x[l:]) / d for l in range(1, k + 1)])
pf = lambda p: "< 0.0001" if p < 1e-4 else f"= {p:.3f}"
def lb(x, h):
    N = len(x); Q = N * (N + 2) * np.sum(acf(x, h) ** 2 / (N - np.arange(1, h + 1))); return Q, 1 - chi2.cdf(Q, h)
print("ACF of returns,         lags 1-3:", np.round(acf(r, 3), 3), " Ljung-Box p", pf(lb(r, 10)[1]))
print("ACF of squared returns, lags 1-3:", np.round(acf(r**2, 3), 3), " Ljung-Box p", pf(lb(r**2, 10)[1]))
def filt(p):
    o, a, b = p; v = np.empty(n); v[0] = r.var()
    for t in range(1, n): v[t] = o + a * r[t-1] ** 2 + b * v[t-1]
    return v
def nll(p):
    o, a, b = p
    if o <= 1e-6 or a < 0 or b < 0 or a + b >= 0.9999: return 1e10
    v = filt(p); return 0.5 * np.sum(np.log(v) + r ** 2 / v)
fit = minimize(nll, [0.1, 0.05, 0.8], method="Nelder-Mead", options={"xatol": 1e-6, "fatol": 1e-8, "maxiter": 4000})
o, a, b = fit.x; v = filt(fit.x)
print(f"GARCH(1,1) estimates: omega {o:.3f} (0.05), alpha {a:.3f} (0.10), beta {b:.3f} (0.85); persistence {a+b:.3f}; long-run variance {o/(1-a-b):.2f}")
z = r / np.sqrt(v)
print("standardized residuals: ACF of squares lags 1-3:", np.round(acf(z**2, 3), 3), " Ljung-Box p", pf(lb(z**2, 10)[1]))
q = 2.326                                           # one-sided 99 percent normal quantile
var_g = q * np.sqrt(v); var_c = q * r.std()
print(f"1-day 99% VaR breaches (expected 1.0%): GARCH {np.mean(r < -var_g)*100:.1f}%, constant-volatility {np.mean(r < -var_c)*100:.1f}%")
calm = v < np.percentile(v, 25); wild = v > np.percentile(v, 75)
print(f"breach rate in calmest quarter of days: constant {np.mean(r[calm] < -var_c)*100:.1f}%, GARCH {np.mean(r[calm] < -var_g[calm])*100:.1f}%")
print(f"breach rate in wildest quarter of days: constant {np.mean(r[wild] < -var_c)*100:.1f}%, GARCH {np.mean(r[wild] < -var_g[wild])*100:.1f}%")

ACF of returns,         lags 1-3: [-0.001  0.023  0.016]  Ljung-Box p = 0.749
ACF of squared returns, lags 1-3: [0.094 0.067 0.126]  Ljung-Box p < 0.0001
GARCH(1,1) estimates: omega 0.054 (0.05), alpha 0.083 (0.10), beta 0.855 (0.85); persistence 0.938; long-run variance 0.87
standardized residuals: ACF of squares lags 1-3: [-0.007 -0.02   0.026]  Ljung-Box p = 0.545
1-day 99% VaR breaches (expected 1.0%): GARCH 1.2%, constant-volatility 1.1%
breach rate in calmest quarter of days: constant 0.5%, GARCH 0.8%
breach rate in wildest quarter of days: constant 2.1%, GARCH 1.1%


## Chapter recap

## Practice set

**1 (easy).** Monthly sales Jan to Mar of one year: 100, 120, 110. Differences? **Answer:** 20 and −10.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/stats/ch10.html